# 経済データ分析Ⅱ　第10回
# トービットモデル

第7回から第9回では、持ち家かどうかや働いていたかどうかのように、0か1の値をとる被説明変数を扱った。今回扱うのは、1年間に働いた時間やたばこへの支出額のように0の観測が多く、0より大きいときにはさまざまな値をとる被説明変数である。

このような被説明変数に OLS を当てはめると何が起こるかを、まず実際のデータで調べる。次に真の係数を決めた人工データを作り、**トービットモデル**（Tobit model、以下、トービット）の考え方と推定の仕組みを学ぶ。最後は実際のデータに戻って OLS と比べよう。

## Notebook の使い方

コードセルを上から順に選び、`Shift + Enter` で実行します。途中のセルから実行して「変数が見つからない」というエラー（`NameError`）が出たら、先頭のセルから実行し直してください。

JupyterLite では **Python (Pyodide)**、ローカルの Jupyter では Python カーネルを使います。第2節のデータはパッケージから読み込み、第4節のデータは Notebook の中で作るので、CSV ファイルの準備は要りません。

## 1. 準備

次のセルで、第9回と同じライブラリを読み込む。

In [ ]:
import sys
if sys.platform == 'emscripten':
    import pyodide_js
    await pyodide_js.loadPackage(['numpy', 'pandas', 'matplotlib', 'statsmodels', 'scipy'])
    import piplite
    await piplite.install('japanize-matplotlib-jlite')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite
import statsmodels.formula.api as smf
from scipy import stats

## 2. 既婚女性の労働時間

第9回の第7節では1975年の米国の既婚女性753人が働いていたかどうかを分析した。同じデータ `mroz` には1975年に働いた時間の合計を表す列 `hours` もあり、働いていなかった女性の `hours` は0になっている。今回はこの労働時間を被説明変数にする。

読み込み方は第9回と同じで、使う列は次の4つ。

- `hours`：1975年に働いた時間（1年間の合計）
- `educ`：教育を受けた年数
- `age`：年齢
- `kidslt6`：6歳未満の子どもの数

In [ ]:
if sys.platform == 'emscripten':
    await piplite.install('wooldridge')
import wooldridge as woo

mroz = woo.dataWoo('mroz')
mroz[['hours', 'educ', 'age', 'kidslt6']].head()

1行目の女性は教育を12年受けた32歳で、6歳未満の子どもが1人いて、1975年に1,610時間働いた。

労働時間が0の女性を数えるには、`mroz['hours'] == 0` で労働時間が0かどうかを True と False で表し、`.sum()` で True の数、`.mean()` で True の割合を求める。`working` には第6回で持ち家の行を取り出したのと同じ書き方で、労働時間が正の女性だけを取り出す。

In [ ]:
zero = mroz['hours'] == 0
working = mroz[mroz['hours'] > 0]
print(f'労働時間が0の女性：{zero.sum()}人（割合{zero.mean():.3f}）')
print(f"労働時間が正の女性：{len(working)}人、平均{working['hours'].mean():.0f}時間")

753人のうち325人、割合で0.432の女性は労働時間が0で、働いた428人の平均は1,303時間だった。1日8時間なら約163日分になる。

第4回でも使った `ax.hist()` で、値の範囲を40の区間（`bins=40`）に分けて労働時間のヒストグラムを描く。

In [ ]:
fig, ax = plt.subplots()
ax.hist(mroz['hours'], bins=40)
ax.set(xlabel='1975年の労働時間（時間）', ylabel='人数')
plt.show()

左端に1本だけ高い棒があり、労働時間が0の325人がここに入る。その右では人数がなだらかに分布し、週40時間を50週働いたときの2,000時間の近くにもう1つの山がある。労働時間は負にならないので働かなかった女性はみな0に集まる。

## 3. OLS による労働時間の回帰

被説明変数が0に集まっていても `smf.ols` で計算はできる。第9回の第7節と同じ3つの説明変数を使って、労働時間を OLS で回帰してみよう。

In [ ]:
ols_m = smf.ols('hours ~ educ + age + kidslt6', data=mroz).fit()
print(ols_m.params.round(2))

教育年数の係数は46.11で、年齢と6歳未満の子どもの数が同じなら、教育年数が1年長い女性は労働時間が約46時間長い。6歳未満の子どもの数の係数は −498.36 で、子どもが1人多いと約500時間短い。教育年数1年分の違いの10倍を超える差である。

第7回の3.3節で線形確率モデルの予測確率が0〜1を外れたように、労働時間の OLS でも当てはめ値が0を下回ることがある。753人の当てはめ値 `ols_m.fittedvalues` で確かめよう。

In [ ]:
fitted = ols_m.fittedvalues
print(f'当てはめ値の最小：{fitted.min():.1f}時間')
print(f'当てはめ値が0未満の女性：{(fitted < 0).sum()}人')

当てはめ値の最小は −514.3 時間で、11人の当てはめ値がありえない負の労働時間になった。

当てはめ値を横軸に、実際の労働時間を縦軸にとった散布図を描く。また、0の位置には第8回でも使った `ax.axhline()` で横線を、`ax.axvline()` で縦線を引いておく。

In [ ]:
fig, ax = plt.subplots()
ax.scatter(fitted, mroz['hours'], s=10)
ax.axhline(0, color='gray', linewidth=0.8)
ax.axvline(0, color='gray', linewidth=0.8)
ax.set(xlabel='OLS の当てはめ値（時間）', ylabel='実際の労働時間（時間）')
plt.show()

縦軸の0の位置には労働時間が0の325人が横一列に並び、その当てはめ値は −514.3 時間から1,200時間近くまで広がっている。OLS は0に集まった観測と正の値に散らばった観測を、1本の1次式で表そうとしている。縦線より左の11個の点では当てはめ値が負になった。

## 4. 潜在変数とトービット

### 4.1. 潜在変数で決まる労働時間

トービットでは第8回の3.2節で説明した潜在変数を使う。第8回では持ち家に住む良さを表す潜在変数 $y^*$ が0より大きい世帯を持ち家としたが、トービットで $y^*$ とするのは、その女性にとって望ましい労働時間である。説明変数を教育年数だけにすると、

$$y_i^*=\beta_0+\beta_1\times教育年数_i+\varepsilon_i$$

である。$\varepsilon_i$ は式に入れていない要因による違いを表し、平均0で標準偏差が $\sigma$ の正規分布に従う。$y_i^*$ が0より大きい女性はその時間だけ働き、0以下の女性は働かないので、観測される労働時間 $y_i$ は

$$y_i=\begin{cases}y_i^* & (y_i^*>0\ のとき)\\ 0 & (y_i^*\le 0\ のとき)\end{cases}$$

になる。$y^*$ が負の女性は働かないことを望んでいると読み、その望みが強いほど $y^*$ は小さい。ただし、−500 の女性も −2000 の女性も観測される労働時間は同じ0である。

真の係数を $\beta_0=-2800$、$\beta_1=250$、$\sigma=1000$ として、この仕組みで500人の人工データを作る。

- `rng.integers(8, 18, size=n)` は、8以上18未満の整数（教育年数）を `n` 個作る。
- `rng.normal(0, 1000, size=n)` は、標準偏差1,000の $\varepsilon$ を作る。
- `np.where(条件, A, B)` は、条件を満たす行では A、満たさない行では B の値をとる。ここでは $y^*$ が正なら $y^*$、それ以外は0になる。

In [ ]:
rng = np.random.default_rng(202611)
n = 500
educ = rng.integers(8, 18, size=n)
eps = rng.normal(0, 1000, size=n)
hours_star = -2800 + 250 * educ + eps
sim = pd.DataFrame({'educ': educ, 'hours_star': hours_star,
                    'hours': np.where(hours_star > 0, hours_star, 0)})
sim.head().round(1)

`hours_star` が潜在変数 $y^*$、`hours` が観測される労働時間 $y$ で、$y^*$ が正の行では2つの列が同じ値に、負の行では `hours` が0になっている。労働時間が0の人数と割合も数えておこう。

In [ ]:
zero_sim = sim['hours'] == 0
print(f'労働時間が0の人：{zero_sim.sum()}人（割合{zero_sim.mean():.3f}）')

500人のうち197人の労働時間が0で、割合の0.394は2節の実際のデータの0.432に近い。

教育年数を横軸にして、$y^*$ を灰色の点、$y$ を色の付いた点で重ねて描き、真の直線 $-2800+250\times教育年数$ も引く。`grid` は直線を描くための横軸の点である。

In [ ]:
grid = np.linspace(8, 17, 100)
fig, ax = plt.subplots()
ax.scatter(sim['educ'], sim['hours_star'], s=10, color='gray', label='潜在変数 y*')
ax.scatter(sim['educ'], sim['hours'], s=10, label='観測される労働時間 y')
ax.plot(grid, -2800 + 250 * grid, color='black', label='真の直線')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set(xlabel='教育年数', ylabel='労働時間（時間）')
ax.legend(loc='upper left')
plt.show()

0より上では色の付いた点が灰色の点の上に重なり、0より下の灰色の点は観測では0の位置に移るので、真の直線の値が小さい教育年数の短い人ほど0に移る点が多くなる。

### 4.2. OLS の推定値の偏り

この人工データで500人全員の OLS と労働時間が正の人だけの OLS を推定し、真の傾き250と比べる。

In [ ]:
ols_all = smf.ols('hours ~ educ', data=sim).fit()
ols_pos = smf.ols('hours ~ educ', data=sim[sim['hours'] > 0]).fit()
pd.DataFrame({'全員の OLS': ols_all.params, '正の人だけの OLS': ols_pos.params,
              '真の値': [-2800, 250]}).round(1)

教育年数の係数は全員の OLS で160.7、正の人だけの OLS で138.9となり、2つとも真の値250より小さい。なぜ小さくなるのだろうか。そこで、第6回で使った `groupby` で教育年数ごとに $y$ と $y^*$ の平均を計算して調べる。

In [ ]:
means = sim.groupby('educ')[['hours', 'hours_star']].mean()
means.round(1).T

教育年数が8年の人では $y^*$ の平均が −1044.2 時間なのに対して、$y$ の平均は49.3時間である。$y^*$ が負の人の $y$ が0に置き換わる分だけ、$y$ の平均が大きくなるからである。一方、17年の人では $y^*$ が負になる人が少なく、$y^*$ の平均1,502.0時間と $y$ の平均1,585.9時間の差は小さい。

教育年数ごとの $y$ の平均を点で描き、真の直線と2つの OLS の直線を重ねる。

In [ ]:
fig, ax = plt.subplots()
ax.scatter(means.index, means['hours'], color='black', label='y の平均')
ax.plot(grid, -2800 + 250 * grid, color='black', label='真の直線')
ax.plot(grid, ols_all.params['Intercept'] + ols_all.params['educ'] * grid, label='全員の OLS')
ax.plot(grid, ols_pos.params['Intercept'] + ols_pos.params['educ'] * grid, label='正の人だけの OLS')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set(xlabel='教育年数', ylabel='労働時間（時間）')
ax.legend(loc='upper left')
plt.show()

$y$ の平均の点は教育年数が短いところで真の直線より上にあり、教育年数が長くなるほど真の直線に近づく。左側だけが持ち上がった点の並びに直線を当てはめれば傾きは緩やかになる。全員の OLS の傾きが真の値より小さいのはこのためである。

正の人だけの OLS の直線は点の並びより上にある。直線が上にあるのは0の197人を除いたからで、教育年数が短い人で労働時間が正になるのは $\varepsilon$ が大きな正の値をとった人だけなので、左側の労働時間が高めになり、傾きもやはり小さくなる。

## 5. トービットの推定

### 5.1. 観測ごとの起こりやすさ

第8回の第4節で説明したように、プロビットの係数はデータで観測された結果が最も起こりやすくなる係数を選ぶ最尤法で推定した。トービットも最尤法で推定する。ただし、statsmodels にはトービットの関数が無いので、参考文献の本と同じく観測ごとの起こりやすさを自分で書くことになる。$y^*$ は平均 $\beta_0+\beta_1\times教育年数$、標準偏差 $\sigma$ の正規分布に従うので、起こりやすさは正規分布で計算できる。

- 労働時間が0の人の起こりやすさは $y^*$ が0以下になる確率で、正規分布の累積分布関数の0での値を `stats.norm.cdf(0, loc=平均, scale=標準偏差)` で計算する。第7回で使った標準正規分布は、`loc=0`、`scale=1` の場合にあたる。
- 労働時間が正の人では $y^*$ が観測された値と一致するが、連続的な値がちょうど1つの値をとる確率は0である。そこで、第7回の2.2節で見た密度関数の $y$ での高さを `stats.norm.pdf(y, loc=平均, scale=標準偏差)` で計算して使う。

真の値 $\beta_0=-2800$、$\beta_1=250$、$\sigma=1000$ で、500人の起こりやすさを計算する。

In [ ]:
mean_star = -2800 + 250 * sim['educ']
like = np.where(sim['hours'] == 0,
                stats.norm.cdf(0, loc=mean_star, scale=1000),
                stats.norm.pdf(sim['hours'], loc=mean_star, scale=1000))
pd.DataFrame({'educ': sim['educ'], 'hours': sim['hours'].round(1),
              'y* の平均': mean_star, '起こりやすさ': like.round(6)}).head()

労働時間が0の行の起こりやすさは確率なので0から1の間の値になり、正の行の値は密度関数の高さなので、標準偏差が1,000と大きいこのデータでは0.000196のように小さくなる。

### 5.2. 対数尤度

500人の起こりやすさを掛け合わせた値を**尤度**（likelihood）という。ただし、小さな値を500個掛けると0に近くなりすぎる。そこで対数をとってから足し合わせた**対数尤度**（log-likelihood）を使い、これが最も大きくなる係数を最尤法の推定値とする。

係数を変えながら同じ計算を何度も行うので手順を関数 `loglik` にまとめておく。`def 名前(受け取る値):` の下に字下げして手順を書き、`return` で結果を返すと、自分で関数を作れる。対数は第2回で人口密度に使った `np.log()` で計算する。

In [ ]:
def loglik(b0, b1, sigma):
    mean_star = b0 + b1 * sim['educ']
    like = np.where(sim['hours'] == 0,
                    stats.norm.cdf(0, loc=mean_star, scale=sigma),
                    stats.norm.pdf(sim['hours'], loc=mean_star, scale=sigma))
    return np.log(like).sum()

print(f'真の値での対数尤度：{loglik(-2800, 250, 1000):.1f}')
print(f"全員の OLS の係数での対数尤度：{loglik(ols_all.params['Intercept'], ols_all.params['educ'], 1000):.1f}")

真の値での対数尤度 −2646.9 は全員の OLS の係数での −2681.7 より大きく、真の値のほうが観測された労働時間をよく説明している。

定数を −2800、$\sigma$ を1,000に固定し、教育年数の係数だけを100から400まで動かして対数尤度を描く。`[loglik(-2800, b, 1000) for b in b1_grid]` は、`b1_grid` の値を1つずつ `b` に入れて計算し、結果を並べる。

In [ ]:
b1_grid = np.linspace(100, 400, 61)
ll_grid = [loglik(-2800, b, 1000) for b in b1_grid]
fig, ax = plt.subplots()
ax.plot(b1_grid, ll_grid)
ax.axvline(250, color='gray', linewidth=0.8)
ax.set(xlabel='教育年数の係数', ylabel='対数尤度')
plt.show()

対数尤度は教育年数の係数が250に近いところで最も大きく、そこから離れるほど小さくなっていく。したがって、最尤法が探すのはこの山の頂上にあたる係数である。

### 5.3. statsmodels による推定

実際の推定では $\beta_0$、$\beta_1$、$\sigma$ の3つを同時に動かして、対数尤度が最も大きくなる値を探す。観測ごとの対数尤度を書いておけば、この探索は statsmodels の `GenericLikelihoodModel` が行ってくれる。そこで、参考文献の本の17.3節と同じく `Tobit` という名前のモデルを作ろう。

- `class Tobit(GenericLikelihoodModel):` は、`GenericLikelihoodModel` の機能を引き継いだ新しいモデル `Tobit` を作る。
- `nloglikeobs` は、係数の値 `params` を受け取り、観測ごとの対数尤度に −1 を掛けた値を返す。statsmodels はこの和を最も小さくする係数、言い換えると対数尤度の和を最も大きくする係数を探す。
- `self.endog` は被説明変数の列、`self.exog` は定数項の列と説明変数の列を並べた表である。`X @ beta` は各行で説明変数の値と係数を掛けて足し合わせたもので、$y^*$ の平均にあたる。
- `params` の最後には $\sigma$ の代わりに $\log\sigma$ を入れ、`np.exp()` で $\sigma$ に戻す。$\log\sigma$ はどんな値をとっても $\sigma$ が正になる。

In [ ]:
from statsmodels.base.model import GenericLikelihoodModel

class Tobit(GenericLikelihoodModel):
    def nloglikeobs(self, params):
        y = self.endog
        X = self.exog
        beta = params[:-1]
        sigma = np.exp(params[-1])
        mean_star = X @ beta
        ll = np.where(y == 0,
                      np.log(stats.norm.cdf(0, loc=mean_star, scale=sigma)),
                      np.log(stats.norm.pdf(y, loc=mean_star, scale=sigma)))
        return -ll

被説明変数の列と説明変数の表は、statsmodels が回帰式を読むときに使うパッケージ `patsy` の `dmatrices` で回帰式の文字から作る。

In [ ]:
import patsy
y, X = patsy.dmatrices('hours ~ educ', data=sim, return_type='dataframe')
X.head()

`X` はすべて1の定数項の列 `Intercept` と教育年数の列 `educ` を並べた表である。

探索を始める値には、全員の OLS の係数の後ろに、OLS の残差の標準偏差の対数を `np.append()` で付け足したものを使う。`extra_params_names` は最後の値 $\log\sigma$ に付ける名前、`maxiter=1000` は繰り返しの計算の上限の回数である。

In [ ]:
start = np.append(ols_all.params, np.log(ols_all.resid.std()))
tobit_sim = Tobit(y, X, extra_params_names=['log_sigma']).fit(start_params=start, maxiter=1000, disp=0)
pd.DataFrame({'推定値': tobit_sim.params, '標準誤差': tobit_sim.bse,
              '真の値': [-2800, 250, np.log(1000)]}, index=tobit_sim.model.exog_names).round(4)

定数項の推定値は −2884.7260、教育年数の係数は257.6079で、真の値 −2800、250 に近い。教育年数の係数の標準誤差18.5111は、第8回の第4節で説明したように推定値のばらつきの大きさを見積もった値で、推定値と真の値の差より大きい。

最後の行の $\log\sigma$ を `np.exp()` で $\sigma$ に戻し、推定値での対数尤度も計算する。

In [ ]:
b_sim = tobit_sim.params
print(f'σ の推定値：{np.exp(b_sim[-1]):.1f}')
print(f'推定値での対数尤度：{loglik(b_sim[0], b_sim[1], np.exp(b_sim[-1])):.1f}')

$\sigma$ の推定値は975.5で、真の値1,000に近い。推定値での対数尤度 −2646.6 が真の値での −2646.9 よりわずかに大きいのは、最尤法が手元のデータで対数尤度を最も大きくする値を選ぶからである。

教育年数の係数は、全員の OLS で160.7、正の人だけの OLS で138.9、トービットで257.6だった。真の値250に近いのはトービットだけで、0に集まった観測を「$y^*$ が0以下になった」確率として扱うためである。

## 6. 既婚女性の労働時間のトービット

2節の既婚女性のデータに戻り、第3節と同じ3つの説明変数でトービットを推定する。手順は5.3節と同じで、変わるのは回帰式の文字と使うデータ、探索を始める値（第3節の OLS の結果）である。`b_t` は推定値に係数の名前を付けた表である。

In [ ]:
formula = 'hours ~ educ + age + kidslt6'
y_m, X_m = patsy.dmatrices(formula, data=mroz, return_type='dataframe')
start_m = np.append(ols_m.params, np.log(ols_m.resid.std()))
tobit_m = Tobit(y_m, X_m, extra_params_names=['log_sigma']).fit(start_params=start_m, maxiter=1000, disp=0)
b_t = pd.Series(tobit_m.params, index=tobit_m.model.exog_names)
sigma_m = np.exp(b_t['log_sigma'])
print(f'σ の推定値：{sigma_m:.1f}')

$\sigma$ の推定値は1,294.2時間になった。続いて全員の OLS、働いた428人だけの OLS、トービットの係数を並べるが、トービットは最後の $\log\sigma$ を除いた4つの係数 `b_t[:-1]` を使う。

In [ ]:
ols_work = smf.ols(formula, data=working).fit()
pd.DataFrame({'全員の OLS': ols_m.params, '働いた人だけの OLS': ols_work.params,
              'トービット': b_t[:-1]}).round(2)

教育年数の係数は全員の OLS で46.11、働いた人だけの OLS で −14.91、トービットで103.58である。働いた人だけの OLS では符号まで変わり、トービットの値は全員の OLS の2倍を超える。4.2節の人工データと同じく、2つの OLS は0に集まった観測の扱いで偏っている。6歳未満の子どもの数の係数も、トービットの −1049.33 は全員の OLS の −498.36 の約2倍である。

トービットの係数は潜在変数 $y^*$ への効果を表すので、年齢と6歳未満の子どもの数が同じなら、教育年数が1年長い女性は望ましい労働時間が約104時間長いと読む。

## 参考文献

- [Heiss, F. and Brunner, D. (2024) Using Python for Introductory Econometrics, 2nd edition](https://www.urfie.net/downloads/PDF/UPfIE_web.pdf)、17.3節

## データの出典

- 第2節、第3節、第6節のデータは、Python の [`wooldridge` パッケージ](https://pypi.org/project/wooldridge/)の `mroz` である。パッケージの説明は、データの出どころを Mroz, T. A. (1987) The Sensitivity of an Empirical Model of Married Women's Hours of Work to Economic and Statistical Assumptions, Econometrica, 55(4), 765–799 としている。

## 関数の資料

- [NumPy：条件で値を選ぶ（`where`）](https://numpy.org/doc/stable/reference/generated/numpy.where.html)
- [SciPy：正規分布（`stats.norm`）](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html)
- [statsmodels：対数尤度を書いて作るモデル（`GenericLikelihoodModel`）](https://www.statsmodels.org/stable/generated/statsmodels.base.model.GenericLikelihoodModel.html)
- [patsy：回帰式から表を作る（`dmatrices`）](https://patsy.readthedocs.io/en/latest/API-reference.html#patsy.dmatrices)